Week 16 · Day 5 — Citations & Source Transparency
Why this matters

An answer is only as credible as its source.
Today you’ll make your Q&A bot more trustworthy by displaying citations — linking each part of the answer to the retrieved chunks that support it. This turns your app from “talkative” to traceable.

Theory Essentials

Citations connect generated text to retrieval results.

Each chunk retrieved should have metadata: doc_id, chunk_id, and text.

You can show citations as superscript numbers ([1], [2]) or expandable text.

Optionally highlight source snippets within the answer for transparency.

If multiple chunks come from the same doc, group them.

Good UX = clear link between answer and origin (no hidden logic).

In [ ]:
# app/backend.py (improved)
import joblib, pandas as pd
from pathlib import Path
from sklearn.neighbors import NearestNeighbors
from transformers import pipeline

VECTOR_DIR = Path("vector_store")

def load_vector_store(vector_dir: Path = VECTOR_DIR):
    obj = joblib.load(vector_dir / "tfidf_index.joblib")
    return obj["vectorizer"], obj["matrix"], obj["nn"], pd.read_parquet(vector_dir / "chunks.parquet")

def retrieve(query, k, vectorizer, nn, chunks):
    q_vec = vectorizer.transform([query])
    dist, idx = nn.kneighbors(q_vec, n_neighbors=k)
    results = chunks.iloc[idx[0]].copy()
    results["distance"] = dist[0]
    return results

def generate_answer(context, query):
    model = pipeline("text2text-generation", model="google/flan-t5-small")
    prompt = f"Answer using only this context:\n{context}\n\nQuestion: {query}"
    return model(prompt, max_length=200)[0]["generated_text"]

def answer_query(query, k=3):
    vectorizer, matrix, nn, chunks = load_vector_store()
    retrieved = retrieve(query, k, vectorizer, nn, chunks)
    context = "\n".join(retrieved["text"].tolist())
    answer = generate_answer(context, query)
    # add citation index mapping
    citations = []
    for i, row in retrieved.iterrows():
        citations.append({
            "ref": f"[{i+1}]",
            "doc_id": row["doc_id"],
            "chunk_id": int(row["chunk_id"]),
            "excerpt": row["text"][:250]
        })
    return {"query": query, "answer": answer, "citations": citations}


This code is for backend.py tehrefore it can't be tested in this notebook.

All answers to exercises are answered in backend.py/frontend.py

1) Core (10–15 min)
Task: Add a button to copy all citations to clipboard (e.g., st.button("Copy citations")).

2) Practice (10–15 min)
Task: Modify the answer output to show citations inline — e.g., after each paragraph.

3) Stretch (optional, 10–15 min)
Task: Highlight text overlaps between retrieved chunks and generated answer using color-coded spans.

Mini-Challenge (≤40 min)

Goal: Add fully functional citation display and improve UI trust.

Acceptance Criteria
✅ Citations visible below each answer
✅ Each citation shows document ID + excerpt
✅ Works for multiple queries in one chat session
✅ Clean UI layout (titles, colors, expanders)

Stretch Goal: Inline [1], [2] markers that match expanded excerpts.

Notes / Key Takeaways

Citations = credibility; users can verify answers.

Retrieval metadata should always include doc + chunk IDs.

Inline markers improve transparency for long answers.

Proper UX (expanders, grouping) keeps UI readable.

Tomorrow, you’ll evaluate your bot’s accuracy and hallucination rate.

Reflection

How could you automate citation placement within generated text?

How might citation quality help debug your retrieval pipeline?

1) How could you automate citation placement within generated text?

Instead of appending [1][2] at the end, you can align each retrieved chunk with the sentences in the generated answer.

One approach: compute similarity between each sentence in the answer and each retrieved chunk, then place the matching citation number right after that sentence.

Alternatively, you can fine-tune or prompt the LLM itself to output answers with citations inline (e.g., “Answer with citations in [brackets] after each claim”).

This reduces manual stitching and makes citations more meaningful.

2) How might citation quality help debug your retrieval pipeline?

If citations are consistently irrelevant, it signals that your retrieval step (embeddings, chunking, cleaning rules) isn’t working well.

High-quality citations show that the system is grounding answers correctly; poor ones highlight whether the issue is with indexing, query encoding, or chunk size.

By reviewing logs of queries + cited chunks, you can pinpoint whether the failure is due to bad preprocessing (e.g., noisy text), embedding weakness (TF-IDF vs dense models), or simply too few chunks.